# Common Voice Thai — EDA แบบง่าย
ดูข้อมูลก่อนเริ่ม train ด้วย pandas โดยตรง ไม่มี helper หรือ pipeline ซับซ้อน

**ชุดที่ใช้:** Common Voice 27.0 ภาษาไทย `cv-corpus-27.0-2026-09-11/th`

- `df` = คลิป **validated** ที่ผ่านกระบวนการตรวจของต้นทาง เป็นตารางหลักที่เราดู
- `df_all` = validated + invalidated + other เพื่อดูภาพรวมคลิปทั้งหมด
- Train/Dev/Test เป็นส่วนย่อยจากต้นทาง **ไม่บวกซ้ำเข้า df_all**
- ส่วน EDA สำรวจ metadata และเปิดเสียงตัวอย่าง; ขั้นเตรียมทดลองท้ายไฟล์คัด 100 คลิปและแบ่ง Train/Dev โดยไม่แก้ข้อมูลดิบและไม่ฝึกโมเดล

**วิธีใช้:** เลือก kernel `.venv` ใน VS Code แล้วกด **Run All** หรือรันทีละเซลล์จากบนลงล่าง

รู้จักคำสั่งก่อน: `head()` ดูตัวอย่าง · `shape` ดูจำนวนแถว/คอลัมน์ · `info()` ดูชนิดข้อมูล · `isna()` ตรวจค่าว่าง · `value_counts()` นับแต่ละค่า

## 1. เปิดข้อมูล
TSV คล้าย CSV แต่ใช้ tab คั่นคอลัมน์ จึงอ่านด้วย `pd.read_csv(..., sep="\t")`

ตอนอ่านเก็บ metadata เป็นข้อความก่อน เพื่อรักษารหัส เช่น client_id ไว้ ค่าว่างและช่องที่มีแต่ whitespace ถือเป็น missing ส่วนค่าที่ต้นทางเขียนไว้ เช่น unknown ให้เห็นตามเดิม

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Audio

# รันได้จาก project root หรือจากโฟลเดอร์ notebook
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("เปิด notebook ภายในโฟลเดอร์ project ก่อน")
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data/raw/common_voice/cv-corpus-27.0-2026-09-11/th"
CLIPS_DIR = DATA_DIR / "clips"
if not (DATA_DIR / "validated.tsv").is_file():
    raise FileNotFoundError(f"ยังไม่พบ validated.tsv: {DATA_DIR}")

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 90)

df = pd.read_csv(
    DATA_DIR / "validated.tsv",
    sep="\t",
    dtype="string",
    keep_default_na=False,
    na_values=[""],
)
df = df.replace(r"^\s*$", pd.NA, regex=True)
print("Data folder:", DATA_DIR)
df.head()

In [ ]:
print("จำนวนคลิปใน df:", f"{df.shape[0]:,}")
print("จำนวนคอลัมน์ต้นทาง:", df.shape[1])
df.columns.tolist()

In [ ]:
# string คือข้อความ; ยังไม่ใช่การอ่าน bytes เสียงลง DataFrame
df.info()

In [ ]:
# สุ่มดู 5 แถว; random_state ทำให้รันซ้ำแล้วเห็นตัวอย่างเดิม
df.sample(5, random_state=42)

## 2. แต่ละคอลัมน์หมายถึงอะไร
หนึ่งแถวของ `validated.tsv` แทนหนึ่งคลิป เสียงจริงอยู่ในไฟล์ MP3 แยกใน `clips/` ไม่ได้อยู่ในคอลัมน์ audio ของตารางนี้

Common Voice เป็นแหล่งบันทึกเสียงมนุษย์และข้อความ ชุดนี้ **ไม่มี label จริง/ปลอมแบบ SEA-Spoof** เราจะสร้าง label ของโครงงานเมื่อเตรียมข้อมูลจริง/TTS ภายหลัง

In [ ]:
column_meanings = {
    "client_id": "รหัสผู้ให้ข้อมูลที่ถูก hash ใช้จัดกลุ่มผู้พูด ไม่ใช่ชื่อจริงหรือหลักฐานยืนยันบุคคล",
    "path": "ชื่อ/เส้นทางไฟล์เสียงใต้ clips/",
    "sentence_id": "รหัสข้อความต้นทาง",
    "sentence": "ข้อความที่ให้ผู้พูดอ่าน ต้องตรวจความตรงกับเสียงเมื่อนำไปใช้จริง",
    "sentence_domain": "หมวดของข้อความ ถ้าต้นทางระบุ",
    "up_votes": "จำนวนโหวตว่าเสียงตรงกับข้อความ",
    "down_votes": "จำนวนโหวตว่าเสียงไม่ตรงกับข้อความ",
    "age": "ช่วงอายุที่ระบุเอง เช่น twenties ไม่ใช่อายุเป็นปีที่แน่นอน",
    "gender": "เพศที่ระบุเอง อาจไม่ระบุ",
    "accents": "สำเนียงที่ระบุเอง อาจไม่ระบุ และไม่ใช่จังหวัด",
    "variant": "รูปแบบย่อยของภาษาที่ระบุไว้ ถ้ามี",
    "locale": "รหัสภาษา/locale เช่น th",
    "segment": "ชื่อส่วนข้อมูลเฉพาะ ถ้าต้นทางจัดไว้",
}
pd.DataFrame({
    "column": df.columns,
    "meaning": [column_meanings.get(col, "ตรวจเอกสารต้นทางเพิ่มเติม") for col in df.columns],
})

## 3. ในแพ็กนี้มีไฟล์อะไรบ้าง
ตารางถัดไปอ่านเพียง header และคอลัมน์แรกเพื่อนับแถว ยังไม่โหลดเสียงทุกคลิป

- validated / invalidated / other = สถานะคลิปสามกลุ่ม
- train / dev / test = ส่วนย่อยสำหรับการฝึกจากต้นทาง อาจซ้อนกับ validated
- reported = รายการคลิปที่ถูกรายงาน ไม่เพิ่มเข้า total อีกครั้ง
- validated_sentences / unvalidated_sentences = รายการ **ข้อความ** ไม่ใช่จำนวนคลิป
- clip_durations = ความยาวคลิปตาม metadata

In [ ]:
file_meanings = {
    "validated.tsv": "คลิปผ่านการตรวจของต้นทาง",
    "invalidated.tsv": "คลิปไม่ผ่านการตรวจของต้นทาง",
    "other.tsv": "คลิปที่ยังไม่จัดเป็น validated/invalidated",
    "train.tsv": "Train ต้นทาง; เป็นส่วนย่อย ไม่บวกเพิ่ม",
    "dev.tsv": "Dev ต้นทาง; เป็นส่วนย่อย ไม่บวกเพิ่ม",
    "test.tsv": "Test ต้นทาง; เป็นส่วนย่อย ไม่บวกเพิ่ม",
    "reported.tsv": "รายการคลิปที่ถูกรายงาน",
    "validated_sentences.tsv": "ข้อความที่ผ่านการตรวจ; ไม่ใช่คลิป",
    "unvalidated_sentences.tsv": "ข้อความที่ยังไม่ผ่านการตรวจ; ไม่ใช่คลิป",
    "clip_durations.tsv": "ความยาวเสียงตาม metadata",
}
file_rows = []
for file in sorted(DATA_DIR.glob("*.tsv")):
    columns = pd.read_csv(file, sep="\t", nrows=0).columns
    first_column = pd.read_csv(file, sep="\t", usecols=[0], dtype="string")
    file_rows.append({
        "file": file.name,
        "rows": len(first_column),
        "columns": len(columns),
        "size_MB": round(file.stat().st_size / 1_000_000, 2),
        "meaning": file_meanings.get(file.name, "ไฟล์ประกอบ"),
    })
files_df = pd.DataFrame(file_rows)
files_df

## 4. ภาพรวมคลิปทั้งหมด
อ่านอีกสองกลุ่ม แล้วเติม `clip_status` **หนึ่งคอลัมน์ที่เราเพิ่มเอง** เพื่อบอกสถานะตอนรวมตาราง ไม่ใช่ฟีเจอร์ที่ป้อนเข้าโมเดล

`df` ยังเป็น validated เหมือนเดิม ตารางรายละเอียดส่วนใหญ่ด้านล่างใช้ df เพื่อไม่ปนกลุ่มที่ไม่ผ่านการตรวจ

In [ ]:
df_invalidated = pd.read_csv(DATA_DIR / "invalidated.tsv", sep="\t", dtype="string", keep_default_na=False, na_values=[""])
df_other = pd.read_csv(DATA_DIR / "other.tsv", sep="\t", dtype="string", keep_default_na=False, na_values=[""])

df_all = pd.concat([
    df.assign(clip_status="validated"),
    df_invalidated.assign(clip_status="invalidated"),
    df_other.assign(clip_status="other"),
], ignore_index=True)
df_all = df_all.replace(r"^\s*$", pd.NA, regex=True)

print("คลิปทั้งหมดตามสามกลุ่ม:", f"{len(df_all):,}")
print("ชื่อไฟล์ไม่ซ้ำ:", f"{df_all['path'].nunique():,}")
status_counts = df_all["clip_status"].value_counts().rename("clips").to_frame()
status_counts["percent"] = (status_counts["clips"] / len(df_all) * 100).round(2)
status_counts

In [ ]:
# nunique() ไม่นับ missing; client_id คือจำนวนรหัส ไม่ยืนยันจำนวนคนจริง
pd.DataFrame({
    "scope": ["all clip buckets", "validated only (df)"],
    "clips": [len(df_all), len(df)],
    "source_columns": [len(column_meanings), len(df.columns)],
    "speaker_keys": [df_all["client_id"].nunique(), df["client_id"].nunique()],
    "sentence_ids": [df_all["sentence_id"].nunique(), df["sentence_id"].nunique()],
    "unique_texts": [df_all["sentence"].nunique(), df["sentence"].nunique()],
})

## 5. ค่าว่างและข้อมูลซ้ำ — เฉพาะ validated
คอลัมน์ที่ว่างทุกแถวอาจหมายถึงต้นทางไม่ได้เก็บรายละเอียดนั้นใน release นี้ ไม่ใช่แปลว่าต้องเดาค่ามาเติม

ข้อความเดียวกันมีหลายคนอ่านได้ จึงไม่ถือว่าข้อความซ้ำทุกแถวเป็นข้อมูลเสีย; ส่วนชื่อไฟล์ซ้ำต้องตรวจเพิ่มก่อนใช้จริง

In [ ]:
missing = pd.DataFrame({
    "missing_rows": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_nonmissing": df.nunique(),
})
missing.index.name = "column"
missing.sort_values("missing_rows", ascending=False)

In [ ]:
pd.Series({
    "duplicate_file_rows_in_validated": df["path"].duplicated().sum(),
    "duplicate_file_rows_across_all_buckets": df_all["path"].duplicated().sum(),
    "repeated_sentence_id_rows_in_validated": df["sentence_id"].duplicated().sum(),
    "repeated_text_rows_in_validated": df["sentence"].duplicated().sum(),
}, name="rows")

## 6. ความยาวเสียง
ใช้ `clip_durations.tsv` ซึ่งหน่วยเป็น milliseconds แล้วหาร 1,000 เป็นวินาที **ยังไม่ใช่การ decode ตรวจ header ทุกไฟล์**

เพิ่ม `duration_s` เองอีกหนึ่งคอลัมน์: df จะมีคอลัมน์ต้นทาง 13 + duration_s ส่วน df_all มี clip_status เพิ่มด้วย

In [ ]:
durations = pd.read_csv(DATA_DIR / "clip_durations.tsv", sep="\t")
assert durations["clip"].is_unique, "ชื่อคลิปในตารางความยาวซ้ำ ต้องตรวจ mapping ก่อน"
durations["duration_s"] = pd.to_numeric(durations["duration[ms]"], errors="coerce") / 1000
duration_map = durations.set_index("clip")["duration_s"]

df["duration_s"] = df["path"].map(duration_map)
df_all["duration_s"] = df_all["path"].map(duration_map)

print("df หลังเพิ่ม duration_s:", df.shape)
print("validated ที่หา duration ไม่ได้:", df["duration_s"].isna().sum())
df[["path", "sentence", "duration_s"]].head(10)

In [ ]:
df["duration_s"].describe(percentiles=[0.5, 0.9, 0.95]).to_frame()

In [ ]:
pd.DataFrame({
    "scope": ["all clip buckets", "validated only"],
    "clips": [len(df_all), len(df)],
    "known_duration_rows": [df_all["duration_s"].notna().sum(), df["duration_s"].notna().sum()],
    "metadata_hours": [df_all["duration_s"].sum() / 3600, df["duration_s"].sum() / 3600],
}).round(3)

In [ ]:
known_duration = df["duration_s"].dropna()
window_counts = []
for seconds in [4.0375, 6.0, 8.0]:
    longer = (known_duration > seconds).sum()
    window_counts.append({
        "window_s": seconds,
        "clips_longer_than_window": longer,
        "percent_of_known_validated": round(longer / len(known_duration) * 100, 2),
    })
pd.DataFrame(window_counts)

ตารางนี้บอกว่ามีกี่คลิปยาวกว่าช่วง input ที่กำหนด ถ้าใช้เฉพาะช่วงแรกก็จะไม่ตรวจช่วงท้าย **ไม่ได้บอกว่า window ใดให้โมเดลแม่นที่สุด** ต้องทดลองบน Dev ภายหลัง และยังไม่เปลี่ยน config ของ AASIST/RawNet2

In [ ]:
ax = df["duration_s"].plot.hist(bins=30, figsize=(8, 3), title="Validated clip duration")
ax.axvline(4.0375, color="red", linestyle="--", label="4.04 s")
ax.axvline(6, color="orange", linestyle="--", label="6 s")
ax.set_xlabel("Duration (seconds)")
ax.set_ylabel("Clips")
ax.legend()
plt.tight_layout()
plt.show()

## 7. ผู้พูด อายุ เพศ สำเนียง — เฉพาะ validated
ตารางเหล่านี้นับ **คลิป** เป็นหลัก ไม่ใช่จำนวนคน รหัส client_id ใช้นับกลุ่มต้นทางได้ แต่ยังไม่ยืนยันว่าแต่ละรหัสเป็นคนไม่ซ้ำจริง

อายุเป็นช่วง เช่น twenties และสำเนียงเป็นค่าที่ระบุเอง ไม่ใช่จังหวัดหรือภูมิลำเนาที่เราตรวจยืนยัน

In [ ]:
df["gender"].fillna("(missing)").value_counts().rename_axis("gender").to_frame("clips")

In [ ]:
df["age"].fillna("(missing)").value_counts().rename_axis("age").to_frame("clips")

In [ ]:
# ดูชื่อค่าที่มีจริงก่อนตีความกลุ่มถิ่น/สำเนียง
df["accents"].fillna("(missing)").value_counts().head(20).rename_axis("accents").to_frame("clips")

In [ ]:
pd.DataFrame({
    "known_clips": df[["age", "gender", "accents"]].notna().sum(),
    "missing_clips": df[["age", "gender", "accents"]].isna().sum(),
    "known_percent": (df[["age", "gender", "accents"]].notna().mean() * 100).round(2),
})

In [ ]:
speaker_counts = df["client_id"].value_counts()
print("รหัสผู้พูดใน validated:", f"{len(speaker_counts):,}")
print("จำนวนคลิปต่อรหัสผู้พูด:")
display(speaker_counts.describe().to_frame("clips_per_key"))
# ไม่แสดงรหัสทั้งหมด; ดูปริมาณข้อมูลที่รหัส 10 อันดับแรกมี
speaker_counts.head(10).reset_index(drop=True).to_frame("clips_for_top_keys")

## 8. ภาษา โหวต และข้อความ
การมีโหวตหรืออยู่ใน validated เป็นข้อมูลจากต้นทาง ไม่ใช่ผลที่เราฟังตรวจทุกคลิปเอง

In [ ]:
display(df["locale"].value_counts(dropna=False).to_frame("clips"))
votes = df[["up_votes", "down_votes"]].apply(pd.to_numeric, errors="coerce")
votes.describe()

In [ ]:
text_lengths = df["sentence"].str.len()
display(text_lengths.describe().to_frame("characters_per_sentence"))
# จำนวนตัวอักษร ไม่ใช่จำนวนคำภาษาไทย
df[["sentence", "sentence_id"]].head(10)

In [ ]:
# เป็นตารางข้อความต้นทางแยกจากคลิป ไม่เพิ่มจำนวนแถวเข้า df
df_sentences = pd.read_csv(DATA_DIR / "validated_sentences.tsv", sep="\t", dtype="string", keep_default_na=False, na_values=[""])
print("validated_sentences:", df_sentences.shape)
df_sentences.head()

## 9. ไฟล์เสียงในเครื่องและตัวอย่าง
ตรวจว่าชื่อไฟล์ใน metadata มีอยู่ใน clips/ หรือไม่ ขั้นนี้ยังไม่ได้ตรวจเนื้อหา/คุณภาพเสียงทั้งชุด จำนวน MP3 บน disk อาจต่างจาก df_all ที่รวมเพียงสามตาราง; เสียงที่ไม่มีแถวในตารางเหล่านี้ยังไม่เลือกเข้าชุดฝึกโดยอัตโนมัติ

อ่าน header ตัวอย่างแบบสุ่ม 5 คลิปเพื่อเห็น sample rate, channels และ duration จริงของตัวอย่าง ไม่เหมารวมว่าเป็นคุณสมบัติของทุกคลิป

In [ ]:
audio_names = {file.name for file in CLIPS_DIR.glob("*.mp3")}
print("ไฟล์ MP3 ใน clips/:", f"{len(audio_names):,}")
print("validated ที่หาไฟล์ไม่พบ:", (~df["path"].isin(audio_names)).sum())
print("all buckets ที่หาไฟล์ไม่พบ:", (~df_all["path"].isin(audio_names)).sum())
print("ไฟล์ MP3 ที่ไม่อยู่ในสามตารางคลิป:", len(audio_names - set(df_all["path"].dropna())))

In [ ]:
import soundfile as sf

sample_rows = df.sample(5, random_state=42)
audio_info_rows = []
for _, row in sample_rows.iterrows():
    path = CLIPS_DIR / row["path"]
    try:
        info = sf.info(path)
        audio_info_rows.append({
            "path": row["path"],
            "sample_rate": info.samplerate,
            "channels": info.channels,
            "header_duration_s": round(info.duration, 3),
            "metadata_duration_s": row["duration_s"],
            "error": None,
        })
    except Exception as error:
        audio_info_rows.append({"path": row["path"], "error": str(error)})
pd.DataFrame(audio_info_rows)

In [ ]:
# เปลี่ยนเลข 0 เป็น 1, 2, ... เพื่อเลือกแถวอื่นใน df
row = df.iloc[0]
print("ข้อความ:", row["sentence"])
print("ไฟล์:", row["path"])
print("ความยาวตาม metadata:", row["duration_s"], "วินาที")
display(Audio(filename=str(CLIPS_DIR / row["path"])))

## ขั้นเตรียมทดลอง 1: รายชื่อเสียงจริงที่มีข้อมูลจำเป็นครบ
เริ่มจาก validated โดยเลือก `client_id` (รหัสจัดกลุ่มผู้พูด), `path` (ชื่อไฟล์เสียง) และ `sentence` (ข้อความสำหรับสร้าง TTS ภายหลัง)

ตัดเฉพาะแถวที่ข้อมูลสามช่องนี้ว่าง ยังไม่ตัดค่าว่างอายุ/เพศ/สำเนียง เพราะไม่ใช่ข้อมูลที่จำเป็นต่อการลอง train ครั้งแรก

`df_candidates` เป็นเพียงรายชื่อที่ข้อมูลครบ ไม่ได้แปลว่าเสียงทุกไฟล์มีคุณภาพดี ขั้นนี้ยังไม่คัด 100 คลิป ไม่แบ่ง Train/Dev ไม่สร้าง TTS และไม่เขียนหรือคัดลอกไฟล์เสียง

In [ ]:
# ทำตารางใหม่ โดยไม่เปลี่ยน df หรือข้อมูลดิบ
df_candidates = df[["client_id", "path", "sentence"]].dropna().copy()

print("จำนวนคลิป validated ก่อนคัด:", len(df))
print("จำนวนคลิปที่มีข้อมูลจำเป็นครบ:", len(df_candidates))
print("จำนวนคลิปที่ตัดออกเพราะข้อมูลจำเป็นว่าง:", len(df) - len(df_candidates))
print("จำนวนรหัสผู้พูด (ไม่ใช่จำนวนคนที่ยืนยันแล้ว):", df_candidates["client_id"].nunique())

df_candidates.head()

## ขั้นเตรียมทดลอง 2: คัดเสียงจริง 100 คลิปและตรวจการอ่านเสียง
ชุดนี้ใช้ลองให้กระบวนการ train ทำงานได้ก่อน ไม่ใช่ชุดทดสอบสุดท้ายหรือชุดสำหรับสรุปผลวิจัย

- สุ่มแบบทำซ้ำได้ด้วย seed 42 แล้วเลือกเพียงหนึ่งคลิปต่อ `client_id` เพื่อให้ชุดเล็กมีรหัสผู้พูดหลากหลาย
- ไม่เลือกข้อความซ้ำ โดยเทียบหลังปรับ Unicode และรวมช่องว่าง; ไม่แก้ข้อความต้นฉบับที่จะใช้กับ TTS
- ไม่กรองอายุ/เพศ/สำเนียงหรือบังคับคลิปยาว 4 วินาทีในขั้นนี้
- อ่านเสียงเต็มคลิป ตรวจความยาว ค่าที่ไม่เป็นตัวเลข และเสียงเงียบสนิท พร้อมเทียบ hash ของเสียงที่ decode แล้วภายใน 100 คลิปนี้
- การอ่านผ่าน **ไม่รับรอง** ว่าเสียงชัด ไม่มี noise หรือข้อความตรงกับเสียงทั้งหมด; ยังต้องฟังตัวอย่าง
- ยังไม่แบ่ง Train/Dev ไม่ resample ไม่ตัด/วนเสียง ไม่สร้าง TTS และไม่คัดลอกไฟล์เสียง

In [ ]:
import unicodedata

PILOT_SIZE = 100
PILOT_SEED = 42

df_pilot = df_candidates.copy()
# เพิ่ม key สำหรับตรวจข้อความซ้ำเท่านั้น ไม่เปลี่ยน sentence เดิม
df_pilot["text_key"] = df_pilot["sentence"].map(
    lambda text: " ".join(unicodedata.normalize("NFC", text).split()).casefold()
)
df_pilot = (
    df_pilot[df_pilot["text_key"].ne("")]
    .sample(frac=1, random_state=PILOT_SEED)
    .drop_duplicates(subset="client_id")
    .drop_duplicates(subset="text_key")
    .head(PILOT_SIZE)
    .reset_index(drop=True)
)

assert len(df_pilot) == PILOT_SIZE, "ข้อมูลไม่พอสำหรับชุดทดลองที่กำหนด"
assert df_pilot["client_id"].is_unique
assert df_pilot["path"].is_unique
assert df_pilot["text_key"].is_unique

print("คลิปที่เลือก:", len(df_pilot))
print("รหัสผู้พูดไม่ซ้ำ:", df_pilot["client_id"].nunique())
print("ข้อความไม่ซ้ำหลัง normalize:", df_pilot["text_key"].nunique())
df_pilot.head()

In [ ]:
import hashlib
import numpy as np
import soundfile as sf

audio_checks = []
for filename in df_pilot["path"]:
    check = {"path": filename, "status": "ok", "error": None}
    try:
        audio, sample_rate = sf.read(
            CLIPS_DIR / filename, dtype="float32", always_2d=True
        )
        if sample_rate <= 0 or len(audio) == 0:
            raise ValueError("เสียงว่างหรือ sample rate ไม่ถูกต้อง")
        if not np.isfinite(audio).all():
            raise ValueError("เสียงมีค่า NaN หรือ infinity")
        peak = float(np.abs(audio).max())
        if peak <= 1e-8:
            raise ValueError("เสียงเงียบสนิทหรือเกือบเป็นศูนย์ทั้งคลิป")

        # เทียบเสียงหลัง decode รวม sample rate และ shape ไม่ใช่ hash ชื่อไฟล์
        digest = hashlib.sha256()
        digest.update(f"{sample_rate}:{audio.shape}".encode("ascii"))
        digest.update(audio.astype("<f4", copy=False).tobytes())
        check.update({
            "sample_rate": sample_rate,
            "channels": audio.shape[1],
            "duration_s": len(audio) / sample_rate,
            "peak": peak,
            "pcm_sha256": digest.hexdigest(),
        })
    except Exception as error:
        check.update({"status": "error", "error": str(error)})
    audio_checks.append(check)

df_audio_checks = pd.DataFrame(audio_checks)
print("ผลตรวจการอ่านเสียง:")
display(df_audio_checks["status"].value_counts().rename_axis("status").to_frame("clips"))
display(df_audio_checks[df_audio_checks["status"].eq("error")][["path", "error"]])

# ถ้าพบปัญหา หยุดก่อนแบ่งข้อมูล ไม่ข้ามหรือลบไฟล์อัตโนมัติ
assert df_audio_checks["status"].eq("ok").all(), "มีเสียงที่ต้องตรวจหรือเปลี่ยนก่อนทำต่อ"
assert df_audio_checks["pcm_sha256"].is_unique, "พบเสียงเหมือนกันภายในชุดทดลอง ต้องตรวจต่อ"

df_pilot = df_pilot.merge(
    df_audio_checks, on="path", how="left", validate="one_to_one"
)
print("เสียงที่ decode เหมือนกันภายในชุดทดลอง:", df_pilot["pcm_sha256"].duplicated().sum())
display(df_pilot[["duration_s", "peak"]].describe())
display(df_pilot.groupby(["sample_rate", "channels"]).size().rename("clips").to_frame())
df_pilot[["path", "sentence", "duration_s", "sample_rate", "channels"]].head()

In [ ]:
# ฟังหนึ่งคลิปจากชุดทดลอง เปลี่ยน 0 เป็น 1, 2, ... เพื่อฟังคลิปอื่น
pilot_row = df_pilot.iloc[0]
print("ข้อความ:", pilot_row["sentence"])
print("ไฟล์:", pilot_row["path"])
print("ความยาวจริง:", round(pilot_row["duration_s"], 3), "วินาที")
display(Audio(filename=str(CLIPS_DIR / pilot_row["path"])))

## ขั้นเตรียมทดลอง 3: แบ่ง Train 80 / Dev 20 และบันทึกรายการ
- **Train:** ข้อมูลที่จะใช้ปรับน้ำหนักโมเดล
- **Dev:** ข้อมูลแยกไว้ดูผลระหว่างพัฒนา/เลือก checkpoint ไม่ใช้ปรับน้ำหนัก
- ขั้นนี้มีเฉพาะเสียงจริง ยัง train ตัวตรวจจับสองคลาสไม่ได้ ต้องสร้างเสียงปลอมต่อ
- เป็น split ใหม่ของ **pilot 100 คลิป** ไม่ใช่ Train/Dev/Test ที่ Common Voice จัดมา และไม่ใช่ Final Test ของงานวิจัย
- ชุดที่เลือกมีหนึ่งคลิปต่อรหัสผู้พูด และข้อความไม่ซ้ำอยู่แล้ว จึงสุ่มแถวได้สำหรับชุดเล็กนี้เท่านั้น; **ห้ามนำวิธีสุ่มแถวนี้ไปใช้กับ corpus เต็มที่ผู้พูด/ข้อความซ้ำ**
- ตรวจรหัสผู้พูด ข้อความตาม `text_key` ชื่อไฟล์ และ decoded PCM hash ว่าไม่ซ้ำข้าม Train/Dev ก่อนบันทึก
- เสียง TTS ที่สร้างจากข้อความใดในขั้นต่อไปต้องอยู่ split เดียวกับข้อความต้นทาง ห้ามสุ่มแบ่งเสียงปลอมใหม่
- บันทึก CSV ใน `data/processed/cvtts/pilot_v1/manifests/` และรายงานใน `qc/`; ไม่มีการคัดลอก/เปลี่ยนไฟล์เสียงดิบ ไฟล์ข้อมูลเหล่านี้ไม่เข้า Git
- หากรันใหม่แล้วผลต่างจากไฟล์ที่บันทึกไว้ โค้ดจะหยุด ไม่เขียนทับ dataset version เดิม

In [ ]:
SPLIT_SEED = 43
TRAIN_SIZE = 80

# df_pilot มาจากขั้น 2 และต้องผ่านการตรวจเสียงแล้ว
assert len(df_pilot) == 100
assert df_pilot["status"].eq("ok").all()
for column in ["client_id", "text_key", "path", "pcm_sha256"]:
    assert df_pilot[column].notna().all() and df_pilot[column].is_unique, column

df_pilot_split = df_pilot.sample(frac=1, random_state=SPLIT_SEED).reset_index(drop=True)
df_pilot_split["split"] = "dev"
df_pilot_split.loc[:TRAIN_SIZE - 1, "split"] = "train"
df_pilot_split["label"] = "bonafide"  # ทั้งชุดนี้ยังเป็นเสียงจริง
df_pilot_split["sample_id"] = df_pilot_split["path"].map(lambda filename: Path(filename).stem)
df_pilot_split["source_dataset"] = "common_voice"
df_pilot_split["source_release"] = "cv-corpus-27.0-2026-09-11"
df_pilot_split["language"] = "th"
# path ใช้ชื่อไฟล์เดิม; audio_path เป็นเส้นทางอ้างอิงจาก project root
df_pilot_split["audio_path"] = df_pilot_split["path"].map(
    lambda filename: (CLIPS_DIR / filename).relative_to(PROJECT_ROOT).as_posix()
)
df_pilot_split["audio_file_sha256"] = df_pilot_split["path"].map(
    lambda filename: hashlib.sha256((CLIPS_DIR / filename).read_bytes()).hexdigest()
)

df_train = df_pilot_split[df_pilot_split["split"].eq("train")].copy()
df_dev = df_pilot_split[df_pilot_split["split"].eq("dev")].copy()

assert len(df_train) == 80 and len(df_dev) == 20
split_checks = []
for column in ["client_id", "text_key", "path", "pcm_sha256", "audio_file_sha256"]:
    overlap = set(df_train[column]) & set(df_dev[column])
    split_checks.append({"check": column, "overlap_train_dev": len(overlap)})
    assert not overlap, f"ข้อมูลซ้ำข้าม Train/Dev: {column}"

display(pd.DataFrame(split_checks))
display(
    df_pilot_split.groupby("split").agg(
        clips=("path", "size"),
        speaker_keys=("client_id", "nunique"),
        texts=("text_key", "nunique"),
        hours=("duration_s", lambda values: values.sum() / 3600),
    )
)
df_train[["sample_id", "sentence", "split", "label"]].head()

In [ ]:
import json

PILOT_DIR = PROJECT_ROOT / "data/processed/cvtts/pilot_v1"
MANIFEST_DIR = PILOT_DIR / "manifests"
QC_DIR = PILOT_DIR / "qc"

# เก็บข้อมูลที่มาจากต้นทางและผลตรวจ ไม่เก็บ status/error ที่ผ่านแล้วทั้งหมด
manifest_columns = [
    "sample_id", "audio_path", "path", "client_id", "sentence", "text_key",
    "split", "label", "source_dataset", "source_release", "language",
    "duration_s", "sample_rate", "channels", "peak",
    "pcm_sha256", "audio_file_sha256",
]
manifest_tables = {
    "real_all.csv": df_pilot_split,
    "real_train.csv": df_train,
    "real_dev.csv": df_dev,
}

# เตรียมเนื้อหาและ hash ก่อนเขียน เพื่อให้ตรวจย้อนหลังและรันซ้ำได้
prepared_files = {}
for filename, table in manifest_tables.items():
    csv_text = table[manifest_columns].sort_values("sample_id").to_csv(
        index=False, lineterminator="\n", float_format="%.10g"
    )
    prepared_files[MANIFEST_DIR / filename] = csv_text.encode("utf-8")

split_report = {
    "dataset_version": "pilot_v1",
    "purpose": "feasibility_only_not_final_test",
    "source_release": "cv-corpus-27.0-2026-09-11",
    "source_pool": "validated.tsv",
    "source_metadata_sha256": hashlib.sha256((DATA_DIR / "validated.tsv").read_bytes()).hexdigest(),
    "selection_seed": PILOT_SEED,
    "selection_rule": "one_clip_per_client_id_and_unique_normalized_text",
    "text_normalization": "Unicode_NFC_whitespace_collapse_casefold_v1",
    "split_seed": SPLIT_SEED,
    "counts": {"train": len(df_train), "dev": len(df_dev)},
    "cross_split_overlap": {item["check"]: item["overlap_train_dev"] for item in split_checks},
    "audio_state": "original_MP3_native_sample_rate_not_resampled",
    "pcm_hash_definition": "native_decoded_float32_LE_with_sample_rate_and_shape",
    "tts_lineage_rule": "generated_audio_inherits_source_text_split",
    "software": {"pandas": pd.__version__, "numpy": np.__version__, "soundfile": sf.__version__},
    "manifest_sha256": {
        path.name: hashlib.sha256(content).hexdigest()
        for path, content in prepared_files.items()
    },
}
prepared_files[QC_DIR / "pilot_split_report.json"] = (
    json.dumps(split_report, ensure_ascii=False, indent=2) + "\n"
).encode("utf-8")

# ตรวจทุกไฟล์ก่อนเขียน ไม่ทับไฟล์เก่าหากผลต่างกัน
for path, content in prepared_files.items():
    if path.exists() and path.read_bytes() != content:
        raise FileExistsError(f"ผลเปลี่ยนจาก dataset version เดิม: {path}; ต้องใช้ version ใหม่")
for path, content in prepared_files.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_bytes(content)
    print("บันทึก/ตรวจตรงกับไฟล์เดิม:", path.relative_to(PROJECT_ROOT).as_posix())

print("เสร็จ: รายการเสียงจริง Train 80 / Dev 20; ยังไม่มีเสียงปลอมและยังไม่ได้ train")

## 10. หลังดู notebook นี้ควรตอบอะไรได้
1. มีไฟล์อะไรบ้าง ต่างกันอย่างไร และอะไรบวกจำนวนรวมได้/ไม่ได้
2. validated มีกี่คลิป มีคอลัมน์อะไร และเสียงอยู่ที่ไหน
3. metadata ช่องใดขาด และอายุ/เพศ/สำเนียงครอบคลุมเพียงใด
4. คลิปยาวประมาณไหน มีกี่คลิปเกิน 4.04 หรือ 6 วินาที
5. จำนวนคลิปต่างจากจำนวน speaker keys และข้อความไม่ซ้ำอย่างไร
6. เสียงตัวอย่างตรงกับข้อความหรือไม่ และ sample rate ต้นทางเป็นเท่าไร

**สิ่งที่ยังไม่ได้ทำ:** ฟังตรวจทั้งชุด, ตรวจ waveform/PCM duplicate ทั้ง corpus, สร้าง Train/Dev/Final Test สำหรับงานวิจัยเต็มรูปแบบ (ขั้นเตรียมทดลอง 3 มีเพียง Train/Dev ของ pilot 100 คลิป), สร้าง TTS, fine-tune หรือสรุปความแม่นยำของโมเดล

ส่วน EDA และขั้นเตรียมทดลอง 1–2 ไม่เขียนไฟล์ข้อมูลใหม่; ขั้น 3 บันทึก CSV รายชื่อเสียงและ JSON รายงานใน `data/processed/cvtts/pilot_v1/` โดยไม่แก้ raw data หากเก็บ notebook ที่มีผลลัพธ์จริง ให้เก็บใน `outputs/` ที่ไม่เข้า Git ส่วน notebook ต้นแบบนี้ควร clear outputs ก่อน commit